In [0]:
from pyspark.sql import functions as F

df = spark.table("retailnew.bronze.customers")

# Flatten nested JSON
df = df.select(
    "customer_id",
    "email",
    "profile.age",
    "profile.gender",
    "profile.location.city",
    "profile.location.state",
    "profile.location.country",
    F.current_timestamp().alias("processed_ts")
)

# Cast types
df = df.withColumn("age", F.col("age").cast("int"))

# Email validation
df = df.filter(F.col("email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"))

# Remove null IDs
df = df.filter(F.col("customer_id").isNotNull())

# Write to Silver
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("retailnew.silver.customers_clean")
)
